# 🧠 The Bias-Variance Tradeoff: A Numerical Decomposition

Welcome to the hands-on explanation notebook for the **Bias-Variance Tradeoff**! In this notebook, we will:
1. Generate a non-linear sinusoidal function with noise.
2. Fit polynomial regression models of varying degrees (1, 4, 15) representing different levels of model complexity.
3. Compute the **mathematical decomposition of bias and variance** from scratch by training models on multiple independent datasets drawn from the same underlying function.
4. Visualize how model predictions fluctuate (variance) vs. how much they miss the true values (bias).
5. Plot the classic **Bias-Variance U-Curve** showing the relationship between complexity and expected test error.
6. Connect these observations to model selection in YOLO (e.g. YOLO Nano vs. YOLO Extra Large).

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Set seed for reproducibility
np.random.seed(42)

## 1. Simulating the True Function and Datasets
Our true function is $f(x) = \cos(1.5 \pi x)$. We will generate multiple training sets of size 20, adding random Gaussian noise $\epsilon \sim \mathcal{N}(0, 0.2^2)$.

In [ ]:
def true_func(x):
    return np.cos(1.5 * np.pi * x)

# Test inputs (to evaluate bias and variance)
x_test = np.linspace(0, 1, 100)
y_true = true_func(x_test)

# Generate one sample training set for visualization
x_train = np.sort(np.random.rand(20))
y_train = true_func(x_train) + np.random.normal(0, 0.2, 20)

plt.figure(figsize=(8, 5))
plt.plot(x_test, y_true, color='black', linewidth=2, label='True Function f(x)')
plt.scatter(x_train, y_train, color='red', edgecolor='k', s=40, label='Noisy Training Set')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Sinusoidal Data Generating Process')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. Visualizing Underfitting vs. Overfitting

Let's fit polynomials of degree 1 (linear), degree 4 (ideal), and degree 15 (overfitting) to our training set.

In [ ]:
degrees = [1, 4, 15]

plt.figure(figsize=(16, 5))
for idx, degree in enumerate(degrees):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x_train[:, np.newaxis], y_train)
    
    y_pred = model.predict(x_test[:, np.newaxis])
    
    plt.subplot(1, 3, idx + 1)
    plt.plot(x_test, y_true, color='black', label='True f(x)')
    plt.plot(x_test, y_pred, color='blue', linewidth=2, label=f'Degree {degree}')
    plt.scatter(x_train, y_train, color='red', edgecolor='k', s=35)
    plt.ylim(-2, 2)
    plt.title(f"Polynomial Degree {degree}")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

-   **Degree 1 (Underfitting):** Simple straight line. High bias (fails to capture the curvature), low variance.
-   **Degree 4 (Sweet Spot):** Captures the curve beautifully. Low bias, low variance.
-   **Degree 15 (Overfitting):** Wiggles wildly to pass through every noisy data point. Low bias on training data, but extremely high variance (test predictions will be very unstable).

## 3. Decomposing Bias and Variance Numerically

To measure Bias and Variance directly, we generate $N = 100$ independent training datasets, fit models on each, and collect their predictions on our test set.
Then we compute:
-   **Average Model Prediction:** $\mathbb{E}[\hat{f}(x)]$
-   **Bias$^2$:** $(\mathbb{E}[\hat{f}(x)] - f(x))^2$
-   **Variance:** $\mathbb{E}[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2]$

In [ ]:
n_datasets = 100
n_samples = 20
degrees_sweep = [1, 2, 4, 8, 15]

# Arrays to store predictions: shape (n_degrees, n_datasets, n_test_points)
all_preds = np.zeros((len(degrees_sweep), n_datasets, len(x_test)))

for d_idx, degree in enumerate(degrees_sweep):
    for i in range(n_datasets):
        x_tr = np.random.rand(n_samples)
        y_tr = true_func(x_tr) + np.random.normal(0, 0.2, n_samples)
        
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        model.fit(x_tr[:, np.newaxis], y_tr)
        
        all_preds[d_idx, i, :] = model.predict(x_test[:, np.newaxis])

Now we compute the bias$^2$, variance, and total expected error across the test inputs, averaging them over the test points.

In [ ]:
bias_sq_results = []
variance_results = []
total_error_results = []

for d_idx, degree in enumerate(degrees_sweep):
    preds_d = all_preds[d_idx, :, :]
    
    mean_pred = np.mean(preds_d, axis=0)
    
    # Bias^2
    bias_sq = np.mean((mean_pred - y_true) ** 2)
    bias_sq_results.append(bias_sq)
    
    # Variance
    variance = np.mean(np.var(preds_d, axis=0))
    variance_results.append(variance)
    
    # Expected squared error: Bias^2 + Variance + Irreducible Noise (0.2^2 = 0.04)
    total_error = bias_sq + variance + 0.04
    total_error_results.append(total_error)

# Plot the Bias-Variance U-Curve
plt.figure(figsize=(10, 6))
plt.plot(degrees_sweep, bias_sq_results, color='red', marker='o', label='Bias² (Underfitting indicator)')
plt.plot(degrees_sweep, variance_results, color='blue', marker='s', label='Variance (Overfitting indicator)')
plt.plot(degrees_sweep, total_error_results, color='black', marker='^', linewidth=2.5, label='Total Expected Error')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('Error Value')
plt.title('Numerical Bias-Variance Tradeoff Curve')
plt.ylim(0, 0.5)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

Observe:
-   As degree increases, **Bias$^2$ drops** rapidly.
-   As degree exceeds 4, **Variance escalates** rapidly.
-   The **Total Expected Error** is U-shaped, reaching its minimum at **Degree 4** (the optimal complexity).

## 💡 Connection to Computer Vision & YOLO
*   **Nano vs. Extra Large Models:** 
    -   `yolo11n` (Nano, 2.6M parameters): Low complexity, high bias. It is fast but may underfit small or complex objects in PTT diagrams.
    -   `yolo11x` (Extra Large, 56.9M parameters): High complexity, low bias, but high variance. If trained on a small dataset (e.g. 50 photos), it will overfit, memorizing background pixels.
*   **Combating Variance (Overfitting):** 
    -   Gather more training data (shifting the U-curve minimum to the right, allowing more complex models).
    -   Regularization: Use weight decay (L2 penalty) or dropout.
    -   Augmentation: Generate flipped, zoomed, and blurred training frames.